# 2층 신경망: 순전파와 역전파 손계산 검증

[Step 5 이슈 #8](https://github.com/ntt65/A2_1/issues/8)의 고정 예제다. 입력 2개 → 은닉 2개 → 출력 1개이며 두 층의 활성함수는 sigmoid, 손실은 **단일 샘플** Binary Cross-Entropy(BCE)다.

먼저 연쇄 법칙과 독립 손계산 기준값을 정리하고 NumPy 결과를 비교한다. 마지막에는 가중치·편향 **9개 전체**에 중심차분을 적용한다. 자동 미분을 사용하지 않는다.

- 손계산 비교: 원소별 `atol=5e-5`, `rtol=0`, 소수점 네 자리 표시도 일치.
- 매개변수 gradient 교차 검증: `h=1e-5`, 최대 절대오차 `<=1e-6`.
- 새 커널에서 위에서 아래로 실행한다. 중간값은 반올림하지 않고 표시할 때만 네 자리로 반올림한다.

In [1]:
# 파일 개요: 고정된 두 층 신경망의 손계산·NumPy·독립 중심차분을 비교하고 결과를 저장한다.
import json
import os
from pathlib import Path
import sys

import numpy as np

repo_root = Path.cwd().resolve()	# 실행 위치를 절대 경로로 고정해 저장소 기준 경로를 찾는다.
if not (repo_root / "src" / "utils" / "math.py").is_file():
    candidate = repo_root.parent	# notebooks 폴더에서 실행했다면 한 단계 위를 후보로 삼는다.
    if repo_root.name == "notebooks" and (candidate / "src" / "utils" / "math.py").is_file():
        repo_root = candidate	# src가 실제로 있는 상위 폴더를 저장소 루트로 선택한다.
    else:
        raise RuntimeError("Run this notebook from the A2_1 root or its notebooks directory")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))	# 현재 저장소의 공통 함수를 가져올 수 있도록 경로를 한 번 추가한다.

from src.utils.math import sigmoid
from src.utils.random import set_seed
from src.calculus import numerical_gradient

set_seed(42)	# 독립 실험의 시작점에서만 공통 난수 시드를 한 번 설정한다.
output_dir = Path(os.environ.get("A2_1_OUTPUT_DIR", str(repo_root / "outputs"))).resolve()	# 검증에서는 임시 경로를 지정할 수 있고 기본 결과는 outputs에 저장한다.
output_dir.mkdir(parents=True, exist_ok=True)	# 출력 디렉터리를 준비하되 기존 파일을 삭제하지 않는다.
print(f"NumPy {np.__version__}; float64; seed=42; fixed input (no random sampling)")

NumPy 2.2.6; float64; seed=42; fixed input (no random sampling)


## 1. 입력과 곱 방향

| 기호 | 값 | shape |
|---|---|---|
| $x$ | $[1,0]^T$ | `(2,)` |
| $W_1$ | $\begin{bmatrix}.1&.2\\.3&.4\end{bmatrix}$ | `(2,2)` |
| $b_1$ | $[0,0]^T$ | `(2,)` |
| $W_2$ | $[.5,.6]$ | `(2,)` |
| $b_2,y$ | $0,1$ | scalar `()` |

NumPy의 1차원 배열에는 행/열 축이 별도로 없지만 수식에서는 $x$를 열벡터로 본다. 첫 층은 반드시 **`W1 @ x + b1`**이다. 출력은 `W2 @ a1 + b2`이며 배치 평균 계수는 없다.

In [2]:
x = np.array([1.0, 0.0], dtype=np.float64)	# 입력 (2,)의 둘째 성분이 0인 고정 예제를 사용한다.
W1 = np.array([[0.1, 0.2], [0.3, 0.4]], dtype=np.float64)	# 행은 은닉 뉴런, 열은 입력 특징인 가중치 (2,2)다.
b1 = np.zeros(2, dtype=np.float64)	# 은닉 뉴런마다 하나의 편향을 두며 처음에는 0이다.
W2 = np.array([0.5, 0.6], dtype=np.float64)	# 두 은닉 활성화를 하나의 출력 logit으로 합치는 가중치다.
b2 = np.float64(0.0)	# 출력층 편향은 벡터가 아닌 스칼라다.
y_true = np.float64(1.0)	# 관측 정답을 1로 고정해 같은 손계산을 재현한다.

assert x.shape == (2,) and W1.shape == (2, 2) and b1.shape == (2,)	# 첫 선형층의 축 의미가 바뀌거나 broadcasting되는 오류를 막는다.
assert W2.shape == (2,) and b2.shape == () and y_true.shape == ()	# 출력 가중치와 스칼라 편향·정답의 계약을 확인한다.


## 2. 순전파를 직접 계산하기

$$\sigma(z)=\frac{1}{1+e^{-z}},\qquad \sigma'(z)=\sigma(z)(1-\sigma(z)).$$

1. $z_1=W_1x+b_1=[.1,.3]^T$ — `(2,)`.
2. $a_1=\sigma(z_1)=[0.5249791874789399861,\;0.5744425168116589872]^T$ — `(2,)`.
3. $z_2=W_2a_1+b_2=.5a_{1,1}+.6a_{1,2}=0.60715510382646538534$ — `()`.
4. $\hat y=\sigma(z_2)=0.64729156999461775097$ — `()`.
5. $L=-[y\ln\hat y+(1-y)\ln(1-\hat y)]=-\ln\hat y=0.43495843685146062506$ — `()`.

로그는 자연로그다. 현재 표본은 $y=1$이고 $0<\hat y<1$이므로 위 확률 표현이 유효하다. 일반적으로 확률이 float64에서 0 또는 1로 포화되면 `log(0)` 문제가 생긴다. $\hat y=\sigma(z_2)$를 대입하여 정리한 동치식

$$L=\log(1+e^{z_2})-yz_2=\operatorname{logaddexp}(0,z_2)-yz_2$$

을 실제 손실 계산에 사용한다. 확률을 임의로 clipping한 다른 함수를 만들지 않는다.

아래 기준 상수는 위 스칼라 식을 독립적으로 고정밀 계산해 옮긴 값이다. NumPy 순전파 결과를 복사하여 기대값을 만드는 방식이 아니다.

In [3]:
hand_reference = {	# NumPy 실행 결과와 독립적으로 계산한 손계산 상수를 그대로 사용한다.
    "z1": np.array([0.1, 0.3]),
    "a1": np.array([0.5249791874789399861, 0.5744425168116589872]),
    "z2": np.float64(0.60715510382646538534),
    "y_pred": np.float64(0.64729156999461775097),
    "L": np.float64(0.43495843685146062506),
    "dL_dy_pred": np.float64(-1.5448988467566710784),
    "dL_dz2": np.float64(-0.35270843000538224903),
    "dL_dW2": np.array([-0.18516458500119814929, -0.20261071823298063973]),
    "dL_da1": np.array([-0.17635421500269112452, -0.21162505800322934942]),
    "dL_dz1": np.array([-0.04397851580869701354, -0.05173350439092561391]),
    "dL_dW1": np.array([[-0.04397851580869701354, 0.0], [-0.05173350439092561391, 0.0]]),
    "dL_db1": np.array([-0.04397851580869701354, -0.05173350439092561391]),
    "dL_db2": np.float64(-0.35270843000538224903),
}

In [4]:
z1 = W1 @ x + b1	# 첫 선형층에서 은닉 logit 두 개를 계산한다.
a1 = sigmoid(z1)	# 각 은닉 logit에 공통 Sigmoid를 원소별로 적용한다.
z2 = W2 @ a1 + b2	# 은닉 활성화의 가중합으로 스칼라 출력 logit을 만든다.
y_pred = sigmoid(z2)	# 출력 logit을 정답 사건의 예측 확률로 바꾼다.
loss = np.logaddexp(0.0, z2) - y_true * z2	# 고정 예제의 BCE를 logit 표현으로 계산한다.
probability_loss = -(y_true*np.log(y_pred) + (1.0-y_true)*np.log1p(-y_pred))	# 같은 손실의 확률 표현을 별도로 계산해 식의 일관성을 확인한다.
np.testing.assert_allclose(loss, probability_loss, atol=1e-15, rtol=0)	# 포화되지 않은 이 예제에서 두 표현의 차이가 허용치 이내인지 확인한다.

forward = {"z1": z1, "a1": a1, "z2": z2, "y_pred": y_pred, "L": loss}
for name, value in forward.items():
    print(f"{name:8} shape={str(np.shape(value)):6} value={value}")

z1       shape=(2,)   value=[0.1 0.3]
a1       shape=(2,)   value=[0.52497919 0.57444252]
z2       shape=()     value=0.6071551038264654
y_pred   shape=()     value=0.6472915699946178
L        shape=()     value=0.43495843685146063


## 3. 연쇄 법칙으로 역전파 유도

출력층에서 입력층 순서로 진행한다. $\odot$는 원소별 곱이다.

**① 예측값에 대한 손실 미분 — scalar `()`**

$$\frac{\partial L}{\partial\hat y}=-\frac{y}{\hat y}+\frac{1-y}{1-\hat y}=-\frac{1}{\hat y}=-1.5448988467566710784.$$

**② 출력 logit에 대한 미분 — scalar `()`**

$$\delta_2=\frac{\partial L}{\partial z_2}
=\frac{\partial L}{\partial\hat y}\hat y(1-\hat y)
=\hat y-y=-0.35270843000538224903.$$

**③ 출력 가중치에 대한 미분 — `(2,)`**

$$\frac{\partial L}{\partial W_2}=\delta_2 a_1
=[-0.18516458500119814929,-0.20261071823298063973].$$

**④ 은닉 활성값에 대한 미분 — `(2,)`**

$$\frac{\partial L}{\partial a_1}=\delta_2 W_2
=[-0.17635421500269112452,-0.21162505800322934942].$$

**⑤ 은닉 logit에 대한 미분 — `(2,)`**

$$\delta_1=\frac{\partial L}{\partial z_1}
=(\delta_2 W_2)\odot a_1\odot(1-a_1)
=[-0.04397851580869701354,-0.05173350439092561391].$$

**⑥ 첫 층 가중치에 대한 미분 — `(2,2)`**

$$\frac{\partial L}{\partial W_{1,ij}}=\delta_{1,i}x_j,\qquad
\frac{\partial L}{\partial W_1}=\delta_1x^T
=\begin{bmatrix}-0.04397851580869701354&0\\-0.05173350439092561391&0\end{bmatrix}.$$

따라서 NumPy에서는 **`np.outer(delta1, x)`**다. 이 고정 입력의 두 번째 성분이0이므로 두 번째 열은0이다.

편향은 덧셈의 미분이1이므로 $\partial L/\partial b_2=\delta_2$ — `()`, $\partial L/\partial b_1=\delta_1$ — `(2,)`다.

`z2`의 네 자리 표시는 **0.6072**, 두 번째 `delta1`은 **−0.0517**이다. 원문 참고 예시의 0.6071/−0.0513을 정답으로 복사하지 않는다([D12](../docs/DECISIONS.md)).

In [5]:
dL_dy_pred = -y_true/y_pred + (1.0-y_true)/(1.0-y_pred)	# BCE를 예측 확률로 직접 미분한 연쇄 법칙의 첫 항이다.
delta2_from_chain = dL_dy_pred * y_pred * (1.0-y_pred)	# 출력 Sigmoid의 도함수를 곱해 logit에 대한 미분을 얻는다.
delta2 = y_pred-y_true	# BCE와 Sigmoid를 합친 식의 기울기 p-y를 독립적으로 계산한다.
np.testing.assert_allclose(delta2_from_chain, delta2, atol=1e-15, rtol=0)	# 중간 미분을 거친 결과와 단순화한 결과가 일치하는지 확인한다.
dW2 = delta2*a1	# 출력 가중치의 기울기는 출력 오차와 은닉 활성화의 곱이다.
da1 = delta2*W2	# 출력 오차를 각 은닉 활성화 방향으로 전달한다.
delta1 = da1*a1*(1.0-a1)	# 은닉 Sigmoid의 도함수는 각 성분에 원소별로 곱한다.
dW1 = np.outer(delta1, x)	# 은닉 오차와 입력의 외적으로 가중치 (2,2)의 축을 유지한다.
db1 = delta1.copy()	# 은닉 편향 기울기를 별도 배열로 복사해 상태 공유를 피한다.
db2 = np.asarray(delta2).copy()	# 출력 편향 기울기의 스칼라 shape와 독립 저장을 유지한다.

backward = {
    "dL_dy_pred": dL_dy_pred, "dL_dz2": delta2, "dL_dW2": dW2,
    "dL_da1": da1, "dL_dz1": delta1, "dL_dW1": dW1,
    "dL_db1": db1, "dL_db2": db2,
}
assert dW2.shape == (2,) and da1.shape == (2,) and delta1.shape == (2,)	# 은닉 활성화와 출력 가중치의 미분 벡터 shape를 확인한다.
assert dW1.shape == (2, 2) and db1.shape == (2,) and db2.shape == ()	# 가중치·편향 각각의 기울기가 원래 매개변수 shape와 맞는지 확인한다.
np.testing.assert_array_equal(dW1[:, 1], np.zeros(2))	# 둘째 입력이 0이므로 해당 가중치 열의 기울기도 정확히 0이어야 한다.


## 4. 손계산과 NumPy 비교

13개 체크포인트의 shape·유한성·원소별 오차를 검사한다. 숫자를 네 자리로 잘라 다음 계산에 재사용하지 않는다.
원래 배열의 `-0.0`과 `0.0`은 같은 수치다. **표시에만** 반올림된0의 부호를 통일하여 `-0.0000`이라는 문자열 차이가 잘못된 불일치를 만들지 않게 한다. 실제 gradient 배열은 변경하지 않는다.

In [6]:
def four_decimal_strings(value):
    """입력 shape를 유지한 소수 네 자리 문자열을 만들고 부호 있는 영은 0.0000으로 표시한다.

    임의 shape의 유한 수치 입력을 받으며 원본을 변경하지 않는 표시 전용 함수다.
    반환 문자열 ndarray는 스칼라 ()를 포함해 입력과 같은 shape를 갖는다.
    비유한 값은 ValueError로 거부하고 실제 계산값의 부호나 정밀도는 바꾸지 않는다.
    """
    array = np.asarray(value, dtype=np.float64)	# 표시 전에 유한성 검사를 할 수 있도록 float64 배열로 읽는다.
    if not np.all(np.isfinite(array)):	# NaN이나 무한대를 정상적인 비교 문자열로 숨기지 않는다.
        raise ValueError("display values must be finite")
    texts = [f"{0.0 if round(float(item), 4) == 0 else float(item):.4f}" for item in array.flat]	# 실제 값은 유지하고 표시할 때만 음의 영을 0.0000으로 통일한다.
    return np.array(texts).reshape(array.shape)	# 펼쳐서 표시한 문자열을 입력과 같은 shape로 되돌린다.


computed = {**forward, **backward}	# 순전파와 역전파의 모든 체크포인트를 한 비교 집합으로 모은다.
hand_errors = {}
assert set(computed) == set(hand_reference)	# 계산 항목이 손계산 기준에서 빠지거나 추가되지 않았는지 확인한다.
for name, expected in hand_reference.items():
    actual = np.asarray(computed[name])	# 비교할 실측값을 원래 shape가 유지되는 배열로 읽는다.
    assert actual.shape == np.shape(expected), name	# 수치가 비슷해도 shape가 다른 결과는 실패 처리한다.
    assert np.all(np.isfinite(actual)), name	# 비유한 결과를 반올림 비교 전에 거부한다.
    np.testing.assert_allclose(actual, expected, atol=5e-5, rtol=0, err_msg=name)	# 독립 손계산 상수와 원시 계산값을 명시한 절대 허용치로 비교한다.
    actual_text = four_decimal_strings(actual)
    expected_text = four_decimal_strings(expected)
    np.testing.assert_array_equal(actual_text, expected_text, err_msg=name)	# 허용오차 검사와 별개로 실제 네 자리 표시가 같은지 확인한다.
    hand_errors[name] = float(np.max(np.abs(actual-expected)))	# 반올림 전 값으로 각 체크포인트의 최대 절대오차를 기록한다.
    print(f"{name:12} shape={str(actual.shape):6} hand={expected_text} NumPy={actual_text}")

max_hand_absolute_error = max(hand_errors.values())	# 전체 체크포인트 중 가장 큰 손계산 차이를 요약한다.
four_decimal_match = True	# 앞의 모든 표시 비교가 통과한 뒤에만 성공 상태를 기록한다.
print(f"All 13 hand-reference checkpoints agree; maximum absolute error={max_hand_absolute_error:.3e}")

z1           shape=(2,)   hand=['0.1000' '0.3000'] NumPy=['0.1000' '0.3000']
a1           shape=(2,)   hand=['0.5250' '0.5744'] NumPy=['0.5250' '0.5744']
z2           shape=()     hand=0.6072 NumPy=0.6072
y_pred       shape=()     hand=0.6473 NumPy=0.6473
L            shape=()     hand=0.4350 NumPy=0.4350
dL_dy_pred   shape=()     hand=-1.5449 NumPy=-1.5449
dL_dz2       shape=()     hand=-0.3527 NumPy=-0.3527
dL_dW2       shape=(2,)   hand=['-0.1852' '-0.2026'] NumPy=['-0.1852' '-0.2026']
dL_da1       shape=(2,)   hand=['-0.1764' '-0.2116'] NumPy=['-0.1764' '-0.2116']
dL_dz1       shape=(2,)   hand=['-0.0440' '-0.0517'] NumPy=['-0.0440' '-0.0517']
dL_dW1       shape=(2, 2) hand=[['-0.0440' '0.0000']
 ['-0.0517' '0.0000']] NumPy=[['-0.0440' '0.0000']
 ['-0.0517' '0.0000']]
dL_db1       shape=(2,)   hand=['-0.0440' '-0.0517'] NumPy=['-0.0440' '-0.0517']
dL_db2       shape=()     hand=-0.3527 NumPy=-0.3527
All 13 hand-reference checkpoints agree; maximum absolute error=2.220e-16


## 5. 전체 매개변수의 독립 수치 검증

매개변수를 다음 순서로 `(9,)` 벡터로 펼친다.

`[W1[0,0], W1[0,1], W1[1,0], W1[1,1], b1[0], b1[1], W2[0], W2[1], b2]`

각 좌표 $i$에서 $[L(\theta+he_i)-L(\theta-he_i)]/(2h)$를 계산한다. 이 경로는 위 역전파식을 사용하지 않고 **순전파 손실만** 평가한다.
`h=1e-5`, `atol=1e-6`, `rtol=0`을 적용한다. 하나의 고정 예제에서 모든 9개 성분을 교차 검증하는 것이며 임의의 신경망/입력에 대한 정확도 보장을 뜻하지 않는다.

In [7]:
parameter_order = ["W1[0,0]", "W1[0,1]", "W1[1,0]", "W1[1,1]",
                   "b1[0]", "b1[1]", "W2[0]", "W2[1]", "b2"]
parameters_flat = np.concatenate((W1.ravel(), b1, W2, [b2]))	# 고정된 순서로 아홉 매개변수를 하나의 벡터에 모은다.
analytical_gradient = np.concatenate((dW1.ravel(), db1, dW2, [db2]))	# 수치 미분과 같은 매개변수 순서로 해석 기울기를 배열한다.


def loss_from_flat(parameters):
    """유한한 float64 매개변수 아홉 개에서 고정 예제의 스칼라 BCE를 계산한다.

    입력 shape는 parameter_order 순서의 (9,)다. 순전파와 안정적인 logit BCE만
    계산하며 역전파 식을 사용하지 않아 중심차분의 독립 기준으로 쓸 수 있다.
    shape가 잘못되거나 비유한 매개변수가 있으면 ValueError를 발생시킨다.
    입력 배열은 변경하지 않는다.
    """
    values = np.asarray(parameters, dtype=np.float64)	# 검산용 매개변수를 float64로 읽되 원본은 바꾸지 않는다.
    if values.shape != (9,) or not np.all(np.isfinite(values)):	# 매개변수 개수와 유한성 계약을 동시에 확인한다.
        raise ValueError("parameters must have shape (9,) with finite values")
    weights1 = values[:4].reshape(2, 2)	# 첫 네 원소를 은닉 가중치의 원래 축으로 복원한다.
    bias1 = values[4:6]	# 다음 두 원소는 은닉 편향이다.
    weights2 = values[6:8]	# 다음 두 원소는 출력 가중치다.
    bias2 = values[8]	# 마지막 원소는 출력 편향 스칼라다.
    hidden = sigmoid(weights1 @ x + bias1)	# 검산은 역전파 값을 재사용하지 않고 순전파부터 새로 계산한다.
    logit = weights2 @ hidden + bias2	# 교란한 매개변수에서 출력 logit을 다시 구한다.
    return float(np.logaddexp(0.0, logit) - y_true*logit)	# 수치 미분기가 비교할 스칼라 손실만 반환한다.


derivative_h = 1e-5	# 중심차분의 양의 간격을 고정해 검산 조건을 명시한다.
numerical_gradient_values = numerical_gradient(loss_from_flat, parameters_flat, h=derivative_h)	# 매개변수를 하나씩 교란해 모든 기울기를 독립 계산한다.
assert analytical_gradient.shape == (9,) and numerical_gradient_values.shape == (9,)
np.testing.assert_allclose(numerical_gradient_values, analytical_gradient, atol=1e-6, rtol=0)	# 해석식과 독립 중심차분을 기존 허용치 그대로 비교한다.
np.testing.assert_array_equal(numerical_gradient_values[[1, 3]], [0.0, 0.0])	# 입력이 0인 두 가중치의 손실 민감도는 수치 검산에서도 0이어야 한다.
max_numerical_gradient_error = float(np.max(np.abs(numerical_gradient_values-analytical_gradient)))	# 아홉 매개변수의 최악 절대오차를 기록한다.
for name, analytic, numeric in zip(parameter_order, analytical_gradient, numerical_gradient_values):
    print(f"{name:9} analytic={analytic: .12f} numerical={numeric: .12f} abs_error={abs(analytic-numeric):.3e}")
print(f"All 9 parameter gradients pass; maximum absolute error={max_numerical_gradient_error:.3e}")

W1[0,0]   analytic=-0.043978515809 numerical=-0.043978515812 abs_error=3.696e-12
W1[0,1]   analytic=-0.000000000000 numerical= 0.000000000000 abs_error=0.000e+00
W1[1,0]   analytic=-0.051733504391 numerical=-0.051733504391 abs_error=5.457e-13
W1[1,1]   analytic=-0.000000000000 numerical= 0.000000000000 abs_error=0.000e+00
b1[0]     analytic=-0.043978515809 numerical=-0.043978515812 abs_error=3.696e-12
b1[1]     analytic=-0.051733504391 numerical=-0.051733504391 abs_error=5.457e-13
W2[0]     analytic=-0.185164585001 numerical=-0.185164584998 abs_error=3.022e-12
W2[1]     analytic=-0.202610718233 numerical=-0.202610718225 abs_error=7.843e-12
b2        analytic=-0.352708430005 numerical=-0.352708430007 abs_error=1.627e-12
All 9 parameter gradients pass; maximum absolute error=7.843e-12


## 6. 확인한 내용

순전파의 곱 방향과 shape, 필수6개 미분과 편향 기울기, $dW_1$의 outer product 구조를 확인했다.
독립 손계산 상수와 NumPy의 원소별 오차·네 자리 표시를 비교했고 모든9개 매개변수를 중심차분으로 다시 검증했다.
검증 결과는 아래 JSON에 원시 정밀도로 저장한다. 사용자 함수 또는 파일 쓰기 오류는 숨기지 않는다.

다음 단계에서는 Gaussian/Bernoulli/categorical likelihood에서 손실함수가 나오는 과정을 별도로 유도한다. 이 노트북은 단일 샘플의 고정 네트워크 검증에 범위를 한정한다.

In [8]:
report = {	# 모든 앞선 assert가 통과한 실측·입력·환경을 같은 보고서 구조로 모은다.
    "seed": 42, "dtype": "float64", "numpy_version": np.__version__,
    "parameter_count": 9, "parameter_order": parameter_order,
    "derivative_h": derivative_h,
    "max_hand_absolute_error": max_hand_absolute_error,
    "max_numerical_gradient_error": max_numerical_gradient_error,
    "four_decimal_match": four_decimal_match, "all_assertions_passed": True,
    "parameters": {name: np.asarray(value).tolist() for name, value in
                   {"x": x, "W1": W1, "b1": b1, "W2": W2, "b2": b2, "y_true": y_true}.items()},
    "forward": {name: np.asarray(value).tolist() for name, value in forward.items()},
    "backward": {name: np.asarray(value).tolist() for name, value in backward.items()},
    "hand_reference": {name: np.asarray(value).tolist() for name, value in hand_reference.items()},
    "shapes": {name: list(np.shape(value)) for name, value in computed.items()},
    "hand_absolute_errors": hand_errors,
    "analytical_gradient": analytical_gradient.tolist(),
    "numerical_gradient": numerical_gradient_values.tolist(),
    "notes": [
        "One sample; no batch mean factor; natural logarithms.",
        "Hand-reference literals are independent of the NumPy forward/backward computation.",
        "Only four-decimal display normalizes signed zero; original arrays are preserved.",
        "The probability BCE and stable logit BCE agree for this nonsaturated fixed example.",
    ],
}
(output_dir / "backprop_metrics.json").write_text(	# 기존 결과 파일명과 인코딩을 유지해 문서 링크가 가리키는 산출물을 저장한다.
    json.dumps(report, ensure_ascii=False, indent=2, allow_nan=False)+"\n",
    encoding="utf-8", newline="\n",
)
print("PASS: 13 hand-reference checkpoints, 4-decimal agreement, all 9 numerical gradients")
print("Saved backprop_metrics.json")

PASS: 13 hand-reference checkpoints, 4-decimal agreement, all 9 numerical gradients
Saved backprop_metrics.json
